# From Clicks to Actions: Spark-Powered Funnel Analysis with LLM-Driven Recommendations
**Team 4** — Shang Chi Hsu, Xiang Li, Ashwini Manokar, Isabel O'Grady, Meenakshi Narendra

> This project repository is created in partial fulfillment of the requirements for the Big Data Analytics course offered by the Master of Science in Business Analytics program at the Carlson School of Management, University of Minnesota.

---

## Purpose of This Notebook

This notebook builds the full funnel intelligence pipeline for an e-commerce platform. It processes large-scale clickstream data using Apache Spark on Databricks, computes funnel metrics at the session, product, brand, and category level, classifies products into behavioral signals, estimates lost revenue, and assembles the CSV outputs consumed by the Streamlit dashboard.

### Notebook Structure
```
0.  Setup & Load Data
0B. Metric Reliability Audit
1.  Global Platform Metrics          (Session CR, User CR, Drop-off)
2.  View: Two Counting Methods       (event-based vs session-dedup)
3.  Category Structure & Benchmark   (Level-2 validation, missing category handling, benchmark map)
4.  Product-level Metrics            (funnel CR, Path A vs B)
5.  Brand-level Metrics
6.  Business Value Metrics           (revenue, price position, price x CR quadrant)
7.  Signal Classification            (situation-aware product signals)
8.  Assemble & Export Dashboard Tables
```


---
## 0. Setup & Load Data

Load raw clickstream data from Databricks Volume and sample 30% of sessions. Session-level sampling (rather than row-level) preserves the integrity of each user journey.

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
import warnings
warnings.filterwarnings('ignore')

In [ ]:
SAMPLE_FRAC = 0.3

cols = [
    'event_time', 'event_type', 'product_id', 'category_id',
    'category_code', 'brand', 'price', 'user_id', 'user_session'
]

spark_df = spark.read.csv(
    'dbfs:/Volumes/msbabigdata/spark/trend_market/2019-Oct.csv.gz',
    header=True,
    inferSchema=True
).select(cols)

sampled_sessions_df = (
    spark_df
    .select('user_session')
    .distinct()
    .sample(fraction=SAMPLE_FRAC, seed=42)
)

spark_sampled = spark_df.join(sampled_sessions_df, on='user_session', how='inner')

df = spark_sampled.toPandas()
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['date'] = df['event_time'].dt.date

print(f'Rows: {len(df):,}  |  Sessions: {df["user_session"].nunique():,}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')

---
## 0B. Metric Reliability Audit

This section classifies every metric by whether it is reliable on the 30% session sample used in this notebook, or requires full data.

### Metric Reliability Classification

| Metric | Sample-safe? | Reason if not |
|--------|-------------|---------------|
| Session CR, User CR, Drop-off | ✅ Safe | Aggregated at session/user level — sampling preserves proportions |
| Product CR (View→Cart, Cart→Purchase, View→Purchase) | ✅ Safe | Session-dedup counts are preserved by session-level sampling |
| Category & Brand CR | ✅ Safe | Same as above |
| Price position, Price volatility, Revenue | ✅ Safe | Product-level aggregates |
| Price × CR Quadrant | ✅ Safe | Derived from safe metrics |
| **Repeat View Index** | ⚠️ Full data only | Session sampling strips most within-session repeat views; index collapses to ~1.0 artificially |
| **Cross-day Repeated Views** | ⚠️ Full data only | Session sampling makes most multi-day return visits invisible per user |
| **Path A vs B (session-level)** | ⚠️ Underestimates Path A | Cart and purchase events may fall in different sessions; cross-session version (§4.3) is used instead |

On full Databricks data, all ⚠️ metrics become reliable.


---
## 1. Global Platform Metrics

These metrics describe the overall health of the platform and are not broken down by product or seller.

**Audience:** Platform operator

### 1.1 Session Conversion Rate (Session CR)

**Definition**
```
Session Purchase CR = sessions with ≥1 purchase event  /  total sessions
Session Cart CR     = sessions with ≥1 cart event      /  total sessions
```

**Business meaning**
If Session Purchase CR = 5%, then 5 out of every 100 visits end with a purchase. A sustained drop is an early warning signal.

**Limitations**
- No `remove_from_cart` events in this dataset — cart dropout is approximated as sessions that have a cart event but no purchase.
- Cross-session behaviour is not captured: a user who carts on Monday and purchases on Wednesday counts as a dropout in Monday's session.


In [ ]:
# ── Session CR ───────────────────────────────────────────────────────────────
total_sessions = df['user_session'].nunique()

sessions_with_purchase = df[df['event_type'] == 'purchase']['user_session'].nunique()
sessions_with_cart     = df[df['event_type'] == 'cart']['user_session'].nunique()

session_purchase_cr = sessions_with_purchase / total_sessions * 100
session_cart_cr     = sessions_with_cart     / total_sessions * 100

print('=== Session Conversion Rate (Global) ===')
print(f'Total sessions         : {total_sessions:,}')
print(f'Sessions with purchase : {sessions_with_purchase:,}  →  Session Purchase CR = {session_purchase_cr:.2f}%')
print(f'Sessions with cart     : {sessions_with_cart:,}  →  Session Cart CR     = {session_cart_cr:.2f}%')
print()

# NOTE: Session Cart CR < Session Purchase CR is expected in this dataset.
# The reason is Path B behaviour: ~55% of purchases happen without any cart event
# in the same session (user goes directly view → purchase).
# This means many purchase sessions contain no cart event at all, so
# sessions_with_purchase > sessions_with_cart is correct — not a bug.
# See §4.3 for the full Path A vs B breakdown.
if session_cart_cr < session_purchase_cr:
    print('NOTE: Session Cart CR < Session Purchase CR — this is expected.')
    print('~55% of purchases in this dataset bypass the cart entirely (Path B).')
    print('Many purchase sessions therefore contain no cart event.')
    print('See §4.3 for the full Path A vs B breakdown.')


### 1.2 User Conversion Rate (User CR)

**Definition**
```
User Purchase CR = users with ≥1 purchase event  /  total unique users
```

**Business meaning**
User CR captures cross-session behaviour that Session CR misses. Comparing the two reveals purchase patterns:
- **Session CR << User CR** → users need multiple visits before buying (high-consideration products, e.g. electronics)
- **Session CR ≈ User CR** → users tend to buy in the same session they browse (impulse purchase)

**Audience:** Platform operator


In [ ]:
# ── User CR ───────────────────────────────────────────────────────────────────
total_users         = df['user_id'].nunique()
users_with_purchase = df[df['event_type'] == 'purchase']['user_id'].nunique()

user_purchase_cr = users_with_purchase / total_users * 100

print('=== User Conversion Rate (Global) ===')
print(f'Total unique users       : {total_users:,}')
print(f'Users with ≥1 purchase   : {users_with_purchase:,}  →  User Purchase CR = {user_purchase_cr:.2f}%')
print()
print(f'Session Purchase CR : {session_purchase_cr:.2f}%')
print(f'User Purchase CR    : {user_purchase_cr:.2f}%')
print()
if user_purchase_cr > session_purchase_cr * 1.5:
    print('Interpretation: User CR is significantly higher than Session CR.')
    print('→ Many users require multiple visits before purchasing (considered purchase behaviour).')
else:
    print('Interpretation: Session CR and User CR are relatively close.')
    print('→ Users tend to buy in the same visit they browse.')

### 1.3 Drop-off Classification

**Definition**
```
converted     = session contains ≥1 purchase event
cart_dropout  = session contains ≥1 cart event, but NO purchase event
view_dropout  = session contains only view events
```

**Business meaning**
- High `view_dropout` → discovery or attraction problem
- High `cart_dropout` → checkout friction (price, shipping, payment options)

**Limitations**
`cart_dropout` is a proxy — it includes users who carted in one session and returned to purchase in a later session. True cart abandonment rate may be lower than measured.


In [ ]:
# ── Drop-off Classification ───────────────────────────────────────────────────
session_events = df.groupby('user_session')['event_type'].apply(set)

def classify_session(events):
    if 'purchase' in events:
        return 'converted'
    elif 'cart' in events:
        return 'cart_dropout'
    else:
        return 'view_dropout'

session_stage = session_events.apply(classify_session)
stage_counts  = session_stage.value_counts()
stage_pct     = session_stage.value_counts(normalize=True) * 100

print('=== Session Drop-off Classification (Global) ===')
for stage in ['view_dropout', 'cart_dropout', 'converted']:
    print(f'  {stage:<15}: {stage_counts.get(stage,0):>8,}  ({stage_pct.get(stage,0):.1f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#B5D4F4', '#FAC775', '#9FE1CB']
labels = ['View dropout\n(saw but never added to cart)',
          'Cart dropout\n(added to cart but did not buy)',
          'Converted\n(completed purchase)']
vals = [stage_pct.get('view_dropout', 0),
        stage_pct.get('cart_dropout', 0),
        stage_pct.get('converted', 0)]
bars = ax.barh(labels, vals, color=colors, height=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11)
ax.set_xlabel('% of all sessions')
ax.set_title('Global Session Drop-off — where users exit the funnel')
ax.set_xlim(0, max(vals) * 1.15)
plt.tight_layout()
plt.show()

---
## 2. View: Two Counting Methods

The word "view" can mean two different things. Using the wrong one produces misleading conversion rates.

### 2.1 Event-based vs Session-deduplicated View Count

| | Event-based | Session-deduplicated |
|---|---|---|
| **What it counts** | Total number of view events | Number of distinct sessions that viewed the product |
| **Code** | `.groupby('product_id').size()` | `.groupby('product_id')['user_session'].nunique()` |
| **Use for** | Traffic volume / exposure heatmap | Calculating CR, drop-off, signals |
| **Do NOT use for** | Conversion rate (denominator inflated by repeat views) | Raw traffic volume |

**Concrete example:**
```
Product X:
  User A → Session 1 (viewed 5 times)
  User A → Session 2 (viewed 3 times)
  User B → Session 3 (viewed 1 time)

Event-based count     = 9
Session-dedup count   = 3

If User B purchased:
  Event-based CR    = 1/9  = 11.1%  <- misleadingly low
  Session-dedup CR  = 1/3  = 33.3%  <- correct: 1 in 3 visits converted
```

The same logic applies to Cart: one session with multiple add/remove actions still counts as 1 cart intent.

**Rule: Always use session-deduplicated counts when calculating any conversion rate.**


In [ ]:
# ── Compare event-based vs session-dedup view counts ─────────────────────────
views_event   = df[df['event_type'] == 'view'].groupby('product_id').size().rename('view_event_count')
views_session = df[df['event_type'] == 'view'].groupby('product_id')['user_session'].nunique().rename('view_session_count')

view_compare = pd.concat([views_event, views_session], axis=1).dropna()
view_compare['ratio'] = (view_compare['view_event_count'] / view_compare['view_session_count']).round(2)

print('=== Event-based vs Session-deduplicated View Counts ===')
print(f'Products with view data   : {len(view_compare):,}')
print()
print('Ratio = event_count / session_count  (how many times on average each session views this product)')
print(view_compare['ratio'].describe().round(2))
print()
print('Products where ratio > 3 (heavy repeat viewing):')
print(f'  {(view_compare["ratio"] > 3).sum():,} products ({(view_compare["ratio"] > 3).mean()*100:.1f}% of all products)')

---
## 3. Category Structure & Benchmark Design

Before calculating product-level metrics, each product must be compared against a meaningful peer group. A smartphone with 3% View→Cart CR looks very different from an iron with the same rate — comparing them against the same global average would be misleading.

### 3.1 Category Code Structure

Category codes follow a two-level hierarchy: `level_1.level_2` (e.g. `electronics.smartphone`). We split them into two columns for benchmark assignment.


In [ ]:
df_cat = df[df['category_code'].notna()].copy()

split = df_cat['category_code'].str.split('.', n=1, expand=True)
df_cat['cat_level_1'] = split[0]
df_cat['cat_level_2'] = split[1].fillna(split[0])

df_cat = df_cat[df_cat['cat_level_1'].notna() & df_cat['cat_level_2'].notna()]

print(f'Tier 1 rows (has category_code): {len(df_cat):,}')
print(f'Unique Level-2 categories: {df_cat["cat_level_2"].nunique()}')

### 3.2 Threshold Constants

Three minimum-view thresholds are used at different levels of aggregation:

| Constant | Value | Level | Rationale |
|---|---|---|---|
| `MIN_VIEWS_BENCHMARK` | 1000 | Category benchmark | Needs enough sessions to produce a stable category average |
| `MIN_PRODUCT_VIEWS` | 50 | Product | Minimum sessions for a product's own CR to be statistically meaningful |
| `MIN_BRAND_VIEWS` | 200 | Brand × category | Between the two — brands aggregate multiple products |
| `CR_RATIO_THRESHOLD` | 2.0 | Benchmark level decision | Level-2 benchmark used when subcategories differ by ≥2x in CR |
| `MIN_BRAND_PRODUCTS` | 3 | Brand table | Minimum actionable products for a brand to appear in the brand table |

Products/brands that do not clear their threshold are excluded from signal detection rather than silently dropped.


In [ ]:
MIN_VIEWS_BENCHMARK = 1000
MIN_PRODUCT_VIEWS   = 50
MIN_BRAND_VIEWS     = 200
CR_RATIO_THRESHOLD  = 2.0
MIN_BRAND_PRODUCTS = 3


### 3.3 Validating Level-2 as the Benchmark Level

**The question:** Should each product be compared against its Level-1 average (e.g. all `electronics`) or Level-2 average (e.g. `electronics.smartphone`)?

**The test:** If Level-2 subcategories within the same Level-1 parent show CR differences of ≥2x, Level-1 benchmarks are too coarse.

**Decision rule:**
- Use Level-2 benchmark when CR ratio ≥ 2.0x within a Level-1 category
- Fall back to Level-1 when a Level-2 subcategory has fewer than 1,000 view sessions (insufficient sample)

**Why ratio instead of absolute spread?**
A 1% spread means very different things across categories. In `apparel` (base CR ~0.8%), a 1% spread is a 2.25x difference — highly meaningful. In `appliances` (base CR ~3.5%), the same 1% spread is only a 29% difference. The ratio captures relative difference regardless of the baseline.

The result is encoded as a joinable `benchmark_map` dataframe used throughout downstream signal detection.


In [ ]:
benchmark_rows = []

for cat1 in sorted(df_cat['cat_level_1'].unique()):
    subset = df_cat[df_cat['cat_level_1'] == cat1]
    l2_groups = subset.groupby('cat_level_2')['event_type'].value_counts().unstack(fill_value=0)

    for col in ['view', 'cart', 'purchase']:
        if col not in l2_groups.columns:
            l2_groups[col] = 0

    l2_groups = l2_groups[l2_groups['view'] >= MIN_VIEWS_BENCHMARK]

    if len(l2_groups) >= 2:
        l2_groups['v2c'] = l2_groups['cart'] / l2_groups['view']
        cr_min = l2_groups['v2c'].min()
        cr_max = l2_groups['v2c'].max()

        if cr_min == 0:
            cr_ratio = None
            use_l2   = False
        else:
            cr_ratio = round(cr_max / cr_min, 2)
            use_l2   = cr_ratio >= CR_RATIO_THRESHOLD
    else:
        cr_ratio = None
        use_l2   = False

    benchmark_rows.append({
        'cat_level_1'    : cat1,
        'use_level_2'    : use_l2,
        'cr_ratio'       : cr_ratio,
        'benchmark_level': 'level_2' if use_l2 else 'level_1',
    })

benchmark_map = pd.DataFrame(benchmark_rows).set_index('cat_level_1')
print('=== benchmark_map ===')
print(benchmark_map.to_string())

### 3.4 Missing Category Code — Three-Tier Handling

~32% of events have no `category_code`. Before discarding them, we classify them into three tiers:

| Tier | Condition | Handling |
|---|---|---|
| Tier 1 | Has `category_code` | Full analysis — benchmark assigned via `benchmark_map` |
| Tier 2 | No `category_code`, but `category_id` exists | Excluded from benchmark analysis |
| Tier 3 | Neither `category_code` nor `category_id` | Excluded entirely |

Only Tier 1 products (`df_cat`) are used for signal classification and dashboard outputs. Silent exclusion would skew results; explicit tiers make the data loss transparent.


---
## 4. Product-level Metrics

Product-level metrics answer: *"Which specific products are underperforming, and at which stage of the funnel?"*

All product-level conversion rates use **session-deduplicated counts** (see §2).

### 4.1 Shared Funnel Function

`calc_funnel_cr()` computes View→Cart, Cart→Purchase, and View→Purchase rates for any grouping.

It adds a `path_b_flag` column marking products where `Cart→Purchase CR > 100%`. This occurs when more sessions result in purchase than in cart — meaning most buyers skip the cart entirely (Path B / direct purchase). For flagged products, `Cart→Purchase CR` is unreliable and `View→Purchase CR` is used as the primary metric instead.


In [ ]:
def calc_funnel_cr(dataframe, group_by_cols):
    result = dataframe.groupby(group_by_cols + ['event_type'])['user_session']\
        .nunique().unstack(fill_value=0)

    for col in ['view', 'cart', 'purchase']:
        if col not in result.columns:
            result[col] = 0

    result = result.rename(columns={
        'view': 'view_sessions',
        'cart': 'cart_sessions',
        'purchase': 'purchase_sessions'
    })

    result['view_to_cart_%'] = (
        result['cart_sessions'] / result['view_sessions'] * 100
    ).round(2)

    result['cart_to_purchase_%'] = (
        result['purchase_sessions'] / result['cart_sessions'].replace(0, np.nan) * 100
    ).round(2)

    result['view_to_purchase_%'] = (
        result['purchase_sessions'] / result['view_sessions'] * 100
    ).round(2)

    result['path_b_flag'] = result['cart_to_purchase_%'] > 100

    return result

product_cr = calc_funnel_cr(df_cat, ['product_id'])
pf = product_cr[product_cr['view_sessions'] >= MIN_PRODUCT_VIEWS].copy()

print(f'Products with >= {MIN_PRODUCT_VIEWS} view sessions: {len(pf):,}')

### 4.2 Product CR

**Definition**
```
Product View→Cart CR     = sessions that carted product X  /  sessions that viewed product X
Product Cart→Purchase CR = sessions that bought product X  /  sessions that carted product X
Product View→Purchase CR = sessions that bought product X  /  sessions that viewed product X
```

**Business meaning**
- Low View→Cart CR → product page fails to convert browsing into intent (images, title, pricing, reviews)
- Low Cart→Purchase CR → user intends to buy but something blocks checkout (shipping cost, payment options, trust)

**Limitations**
- Minimum view threshold (`MIN_PRODUCT_VIEWS`) applied — products with too few sessions produce noisy CR estimates.
- `path_b_flag = True` products: use View→Purchase CR, not Cart→Purchase CR.
- Always compare within the same Level-2 benchmark group.


### 4.3 Funnel Path A vs B — Cross-Session Classification

**Definition**
```
Path A (standard): user carted product X at any point before purchasing it
                   (cart and purchase may be in different sessions)
Path B (direct):   user purchased product X with no prior cart event at all
```

**Business meaning**
Path B products are typically impulse buys or repeat purchases by returning customers. A high Path B rate in a category means the cart is not the primary purchase path — and low View→Cart CR in that category is expected behaviour, not a problem.

**Why cross-session instead of session-level?**
Session-level classification misclassifies cases where a user carts in Session 1 and purchases in Session 2 (counting as Path B at the session level, even though the user did use the cart). Cross-session classification at the user-product level is more accurate.


---
## 5. Brand-level Metrics

A product may underperform not because of its own page quality, but because of brand perception. If all products from Brand X have lower-than-average CR in the same category, the issue is brand-level — not product-level.

**Important rule:** Brand CR must always be calculated **within the same Level-2 category**. A brand's overall CR across categories (mixing smartphones and washing machines) is not meaningful.

### 5.1 Missing Brand

~14% of events have no brand. Brands that are frequently untagged will have underrepresented metrics. The `MIN_BRAND_VIEWS` threshold (200) partially mitigates this by requiring sufficient observed sessions.

### 5.2 Brand CR (within Level-2 Category)

**Definition**
```
Brand View→Cart CR = sessions that carted any product from Brand X in Category Y
                     ─────────────────────────────────────────────────────────────
                     sessions that viewed any product from Brand X in Category Y
```

**Business meaning**
Allows a seller to benchmark their brand's funnel performance against competitors in the same space.

**Audience:** Seller (competitive positioning within a category)

**Limitations**
- 14% missing brand rate means some brands are underrepresented.
- A brand selling in many sub-categories will have different CRs per category — these should not be averaged together.


---
## 6. Business Value Metrics

CR alone does not tell the full story. A product with 15% CR at $5 contributes far less revenue than a product with 3% CR at $500. This section adds the business value layer.

### 6.1 Revenue Contribution (within Level-2 Category)

Revenue ranking is scoped within Level-2 category. Cross-category revenue comparison (electronics vs accessories) is not meaningful.

**Definition**
```
Product Revenue = SUM of price at the time of each purchase event
```

**Limitations:** We have price, not cost. Revenue ≠ profit. This is a proxy for business contribution. High-revenue products may also have return rates not captured in this dataset.


In [ ]:
purchases = df[(df['event_type'] == 'purchase') & (df['price'] > 0)].copy()

revenue = purchases.groupby('product_id').agg(
    purchase_count     = ('event_type', 'count'),
    avg_purchase_price = ('price', 'mean'),
    total_revenue      = ('price', 'sum')
).round(2)

product_cat_map = df_cat[['product_id', 'cat_level_1', 'cat_level_2']]\
    .drop_duplicates('product_id').set_index('product_id')

product_full = pf.join(revenue, how='left').fillna(0).join(product_cat_map, how='left')

product_full['revenue_rank_in_category'] = product_full.groupby('cat_level_2')['total_revenue']\
    .rank(ascending=False, method='dense').astype(int)

product_full['revenue_percentile_in_category'] = product_full.groupby('cat_level_2')['total_revenue']\
    .rank(pct=True).mul(100).round(1)

print(f'product_full rows: {len(product_full):,}')

### 6.2 Price Position (within Level-2 Category)

**Definition**
```
Price position = percentile rank of the product's median view-time price
                 within its Level-2 category
```

Comparing a product's price against the entire platform is meaningless — it mixes $3 accessories with $1,500 laptops.

**Limitations:** Prices fluctuate across the observation window; median is used to reduce the impact of outliers.


In [ ]:
product_price = df_cat[df_cat['event_type'] == 'view'].groupby('product_id')['price']\
    .median().rename('median_view_price')

product_cat = df_cat[['product_id', 'cat_level_2']].drop_duplicates('product_id').set_index('product_id')

price_with_cat = product_price.to_frame().join(product_cat)
price_with_cat = price_with_cat[price_with_cat['median_view_price'] > 0]

price_with_cat['price_percentile'] = price_with_cat.groupby('cat_level_2')['median_view_price']\
    .rank(pct=True).mul(100).round(1)

print(f'price_with_cat rows: {len(price_with_cat):,}')

### 6.3 Price × CR Quadrant

Classify each product into one of four quadrants based on price position and View→Purchase CR, both relative to their Level-2 category median.

```
                      CR above median
                            |
  High price                |                High price
  High CR                   |                Low CR
  ──────────────────────────┼──────────────────────────
  Brand premium works       |                Price is a barrier
  Consider raising price    |                Reduce price or add value
                            |
  Low price                 |                Low price
  High CR                   |                Low CR
  ──────────────────────────┼──────────────────────────
  Strong value product      |                Fundamental product issue
  Consider price increase   |                Review everything
                            |
                      CR below median
```

**Audience:** Seller (strategic pricing decisions)


In [ ]:
product_quad = pf[['view_sessions', 'view_to_purchase_%']].join(
    price_with_cat[['median_view_price', 'price_percentile', 'cat_level_2']], how='inner'
)

l2_median_cr = product_quad.groupby('cat_level_2')['view_to_purchase_%'].median()
product_quad['l2_median_cr'] = product_quad['cat_level_2'].map(l2_median_cr)

def assign_quadrant(row):
    high_price = row['price_percentile'] >= 50
    high_cr    = row['view_to_purchase_%'] >= row['l2_median_cr']
    if high_price and high_cr:       return 'High Price / High CR'
    elif high_price and not high_cr: return 'High Price / Low CR'
    elif not high_price and high_cr: return 'Low Price / High CR'
    else:                            return 'Low Price / Low CR'

product_quad['quadrant'] = product_quad.apply(assign_quadrant, axis=1)
print('=== Quadrant distribution ===')
print(product_quad['quadrant'].value_counts())

---
## 7. Signal Classification

This section classifies every product into a behavioral signal using a **situation-aware framework**. The core insight is that the same metric value can mean completely different things depending on how users in that category typically purchase.

### 7.1 Category Situation Assignment

For each category we identify: (1) which funnel stage has the largest gap vs the platform median, and (2) whether users in that category tend to buy directly (high Path B) or through the cart (low Path B). The combination determines the **situation**.

**The Three Situations + Healthy:**

| Situation | Path B | Primary bottleneck | Primary metric | Interpretation |
|---|---|---|---|---|
| **Healthy** | any | None — at or above median | — | No action needed |
| **1: Cart barrier** | Low | View→Cart gap > Cart→Purchase gap | `view_to_cart_%` | Cart is a real intent signal here — product page is failing to convert interest into intent |
| **2: Discovery/appeal problem** | High | View→Purchase below median | `view_to_purchase_%` | Users buy directly without cart — low v2p means product lacks visibility or appeal |
| **3: Checkout friction** | Low | Cart→Purchase gap ≥ View→Cart gap | `cart_to_purchase_%` | Users add to cart but something blocks checkout (shipping, payment, trust) |

**Why the same number can mean different things:**

| Example | Path B | View→Cart | Diagnosis | Action |
|---|---|---|---|---|
| `electronics.smartphone` | 34% | Low | Cart barrier — product page problem | Improve images, pricing, reviews |
| `apparel.shoes` | 85% | Low | Expected — shoes are bought directly | Focus on v2p, not v2c |


In [ ]:
# Stage classification per session
session_events = df_cat.groupby('user_session')['event_type'].apply(set)

def classify_session(events):
    if 'purchase' in events and 'cart' in events: return 'converted_via_cart'
    if 'purchase' in events:                       return 'converted_direct'
    if 'cart' in events:                           return 'cart_dropout'
    return 'view_only'

session_stage = session_events.apply(classify_session).rename('session_stage')
df_cat_staged = df_cat.join(session_stage, on='user_session')

def count_sessions_by_stage(group):
    sessions = group.groupby('user_session')['session_stage'].first()
    return pd.Series({
        'view_sessions'      : group['user_session'].nunique(),
        'cart_sessions'      : (sessions.isin(['cart_dropout', 'converted_via_cart'])).sum(),
        'converted_via_cart' : (sessions == 'converted_via_cart').sum(),
        'converted_direct'   : (sessions == 'converted_direct').sum(),
    })

category_bottleneck = (
    df_cat_staged
    .groupby(['cat_level_1', 'cat_level_2'])
    .apply(count_sessions_by_stage)
)

category_bottleneck['total_purchase_sessions'] = (
    category_bottleneck['converted_via_cart'] +
    category_bottleneck['converted_direct']
)

category_bottleneck['v2c_pct'] = (
    category_bottleneck['cart_sessions'] /
    category_bottleneck['view_sessions'] * 100
).round(2)

category_bottleneck['c2p_pct'] = (
    category_bottleneck['converted_via_cart'] /
    category_bottleneck['cart_sessions'].replace(0, np.nan) * 100
).round(2)

category_bottleneck['path_b_rate'] = (
    category_bottleneck['converted_direct'] /
    category_bottleneck['total_purchase_sessions'].replace(0, np.nan) * 100
).round(2)

category_bottleneck['v2p_pct'] = (
    category_bottleneck['total_purchase_sessions'] /
    category_bottleneck['view_sessions'] * 100
).round(2)

cat_bottleneck_filtered = category_bottleneck[
    category_bottleneck['view_sessions'] >= MIN_VIEWS_BENCHMARK
].copy()

print(f'Categories analysed: {len(cat_bottleneck_filtered)}')
print()
print(cat_bottleneck_filtered[['v2c_pct', 'c2p_pct', 'path_b_rate']].to_string())

In [ ]:
cat_median_v2c = cat_bottleneck_filtered['v2c_pct'].median()
cat_median_c2p = cat_bottleneck_filtered['c2p_pct'].median()
cat_median_v2p = cat_bottleneck_filtered['v2p_pct'].median()
path_b_median  = cat_bottleneck_filtered['path_b_rate'].median()

cat_bottleneck_filtered['v2c_gap'] = (cat_median_v2c - cat_bottleneck_filtered['v2c_pct']).round(2)
cat_bottleneck_filtered['c2p_gap'] = (cat_median_c2p - cat_bottleneck_filtered['c2p_pct']).round(2)
cat_bottleneck_filtered['v2p_gap'] = (cat_median_v2p - cat_bottleneck_filtered['v2p_pct']).round(2)

print(f'Path B median : {path_b_median:.1f}%')
print(f'v2c median    : {cat_median_v2c:.2f}%')
print(f'c2p median    : {cat_median_c2p:.2f}%')
print(f'v2p median    : {cat_median_v2p:.2f}%')

def assign_situation(row, path_b_threshold):
    high_path_b = row['path_b_rate'] >= path_b_threshold

    if high_path_b:
        # 習慣直接買的類別 → 用 v2p 判斷
        if row['v2p_gap'] <= 0:
            return 'Healthy'
        else:
            return 'Situation 2: Discovery/appeal problem'
    else:
        # 習慣用購物車的類別 → 用 v2c vs c2p 判斷
# 習慣用購物車的類別 → 用 v2c vs c2p 判斷
        if row['v2c_gap'] <= 0 and row['c2p_gap'] <= 0:
            return 'Healthy'
        elif row['v2c_gap'] > row['c2p_gap']:
            return 'Situation 1: Cart barrier'
        else:
            return 'Situation 3: Checkout friction'

cat_bottleneck_filtered['situation'] = cat_bottleneck_filtered.apply(
    assign_situation, axis=1, path_b_threshold=path_b_median
)

# primary_bottleneck 只對 path_b 低的類別有意義
cat_bottleneck_filtered['primary_bottleneck'] = cat_bottleneck_filtered.apply(
    lambda r: 'View->Cart' if r['v2c_gap'] > r['c2p_gap'] else 'Cart->Purchase',
    axis=1
)
cat_bottleneck_filtered['bottleneck_gap'] = (
    cat_bottleneck_filtered['v2c_gap'] - cat_bottleneck_filtered['c2p_gap']
).round(2)

print()
print('=== Situation distribution ===')
print(cat_bottleneck_filtered['situation'].value_counts())
print()
print(cat_bottleneck_filtered[[
    'v2c_pct', 'v2c_gap', 'c2p_pct', 'c2p_gap',
    'v2p_pct', 'v2p_gap', 'path_b_rate', 'situation'
]].sort_values('situation').to_string())

### 7.2 Situation Config & Action Mapping

Each situation maps to a primary metric, recommended focus area, and suggested actions.


In [ ]:
situation_config = {
    'Situation 1: Cart barrier': {
        'primary_metric'   : 'view_to_cart_%',
        'secondary_metric' : 'view_to_purchase_%',
        'priority_focus'   : 'Product page quality (images, pricing, trust)',
        'action'           : 'Improve product page — better images, clearer pricing, add reviews'
    },
    'Situation 2: Discovery/appeal problem': {
        'primary_metric'   : 'view_to_purchase_%',
        'secondary_metric' : 'view_sessions',
        'priority_focus'   : 'Product visibility and appeal',
        'action'           : 'Increase exposure — search ranking, recommendations, promotions'
    },
    'Situation 3: Checkout friction': {
        'primary_metric'   : 'cart_to_purchase_%',
        'secondary_metric' : 'view_to_cart_%',
        'priority_focus'   : 'Checkout experience (shipping, payment, trust)',
        'action'           : 'Reduce checkout friction — review shipping cost, simplify flow, add payment options'
    },
}

config_df = pd.DataFrame(situation_config).T
config_df.index.name = 'situation'

category_priority = cat_bottleneck_filtered.join(config_df, on='situation')

In [ ]:
signal_action_map_v2 = {
    'Low Cart Rate — fix product page': {
        'priority_focus': 'Product Page Quality',
        'recommended_actions': 'Review product images | Check title and description | Compare price vs competitors | Check reviews/ratings',
        'business_hypothesis': 'Users view but do not add to cart. In this category, cart intent is the key purchase signal. The product page is likely failing to convert curiosity into intent.'
    },
    'Low Direct Purchase Rate — fix visibility': {
        'priority_focus': 'Product Visibility and Appeal',
        'recommended_actions': 'Boost search ranking | Add to recommended sections | Run targeted promotions | Check findability',
        'business_hypothesis': 'In this category, most purchases happen directly without cart (Path B). Low view-to-purchase rate suggests the product is not attracting or converting the users who do find it.'
    },
    'Low Checkout Rate — fix checkout': {
        'priority_focus': 'Checkout Friction',
        'recommended_actions': 'Review shipping cost | Check checkout steps | Consider limited-time discount | Verify payment options',
        'business_hypothesis': 'Users add to cart but do not complete purchase. Something at the checkout stage is blocking conversion.'
    },
    'Low Overall CR — review traffic quality': {
        'priority_focus': 'Traffic Quality and Product-Market Fit',
        'recommended_actions': 'Review traffic sources | Check audience targeting | Verify category placement | Consider product-market fit',
        'business_hypothesis': 'Both cart and purchase rates are low despite Path B being common. The product may not be reaching the right audience.'
    },
    'High Direct Purchase (Path B)': {
        'priority_focus': 'Increase Exposure',
        'recommended_actions': 'Increase promotion and visibility | Feature in banners | Consider paid placement | Expand audience',
        'business_hypothesis': 'Strong converter with loyal or returning buyers. The opportunity is volume, not conversion improvement.'
    },
    'Strong Performer': {
        'priority_focus': 'Maintain and Monitor',
        'recommended_actions': 'No immediate action required | Monitor CR trend | Benchmark periodically',
        'business_hypothesis': 'Product is performing above the 75th percentile on its category primary metric.'
    },
    'Normal': {
        'priority_focus': 'Maintain and Monitor',
        'recommended_actions': 'No immediate action required | Monitor CR trend | Benchmark periodically',
        'business_hypothesis': 'Product is performing within expected range for its category.'
    },
    'Underperformer': {
        'priority_focus': 'Review Category Assignment',
        'recommended_actions': 'Check category_code assignment | Verify sufficient view sessions',
        'business_hypothesis': 'Product situation could not be determined — category may not have enough data.'
    },
    'Insufficient data': {
        'priority_focus': 'Insufficient Data',
        'recommended_actions': 'Wait for more traffic | Check if newly listed',
        'business_hypothesis': 'Not enough sessions to compute a reliable percentile rank.'
    },
}

signal_action_df_v2 = pd.DataFrame([
    {
        'signal_v2'             : signal,
        'priority_focus_v2'     : info['priority_focus'],
        'recommended_actions_v2': info['recommended_actions'],
        'business_hypothesis_v2': info['business_hypothesis']
    }
    for signal, info in signal_action_map_v2.items()
]).set_index('signal_v2')

print('signal_action_df_v2 built.')

### 7.3 Product Signal Assignment

Each product is assigned a signal based on its situation and its percentile rank on the situation's primary metric.

**Signal labels:**

| Signal | Condition | Action type |
|---|---|---|
| `Strong Performer` | Primary metric ≥ 75th percentile | Monitor |
| `Normal` | 25th–75th percentile | Monitor |
| `Low Cart Rate — fix product page` | Situation 1, ≤ 25th percentile | Fix |
| `Low Direct Purchase Rate — fix visibility` | Situation 2, ≤ 25th percentile | Fix |
| `Low Checkout Rate — fix checkout` | Situation 3, ≤ 25th percentile | Fix |
| `High Direct Purchase (Path B)` | `path_b_flag = True` | Scale |
| `Underperformer` | Category join failed | Review |
| `Insufficient data` | Too few sessions to rank | Wait |


In [ ]:
cat_info = df_cat[['product_id', 'cat_level_1', 'cat_level_2']]\
    .drop_duplicates('product_id').set_index('product_id')

product_signals = pf.copy()
product_signals = product_signals.join(cat_info)

cat_config_cols = [
    'situation', 'primary_metric', 'secondary_metric',
    'priority_focus', 'action',
    'path_b_rate',
    'v2c_pct',
    'c2p_pct',
    'v2p_pct',
]
product_signals = product_signals.join(
    category_priority[cat_config_cols],
    on=['cat_level_1', 'cat_level_2'],
    how='left'
)

print(f'Products with situation assigned: {product_signals["situation"].notna().sum():,}')
print(f'Products without situation      : {product_signals["situation"].isna().sum():,}')

In [ ]:
product_signals = product_signals.join(
    benchmark_map[['benchmark_level']], on='cat_level_1', how='left'
)

benchmark_group_key = product_signals.apply(
    lambda r: r['cat_level_2'] if r.get('benchmark_level') == 'level_2'
              else r['cat_level_1'],
    axis=1
)

def assign_primary_percentile(group):
    group = group.copy()
    primary = group['primary_metric'].iloc[0]
    if primary not in group.columns:
        group['primary_percentile'] = np.nan
        return group
    group['primary_percentile'] = (
        group[primary]
        .rank(pct=True)
        .mul(100)
        .round(1)
    )
    return group

product_signals = (
    product_signals
    .assign(benchmark_group=benchmark_group_key)
    .groupby('benchmark_group', group_keys=False)
    .apply(assign_primary_percentile)
)

product_signals = product_signals.rename(columns={
    'v2c_pct': 'cat_v2c_benchmark',
    'c2p_pct': 'cat_c2p_benchmark',
    'v2p_pct': 'cat_v2p_benchmark',
})

print('Percentile ranking done.')

In [ ]:
LOW_PCT  = 25
HIGH_PCT = 75

def assign_signal_v2(row):
    if row.get('path_b_flag'):
        return 'High Direct Purchase (Path B)'

    percentile = row.get('primary_percentile')
    if pd.isna(percentile):
        return 'Insufficient data'

    situation = row.get('situation', '')

    if percentile <= LOW_PCT:
        if situation == 'Situation 1: Cart barrier':
            return 'Low Cart Rate — fix product page'
        elif situation == 'Situation 2: Discovery/appeal problem':
            return 'Low Direct Purchase Rate — fix visibility'
        elif situation == 'Situation 3: Checkout friction':
            return 'Low Checkout Rate — fix checkout'
        elif situation == 'Situation 4: Traffic quality problem':
            return 'Low Overall CR — review traffic quality'
        else:
            return 'Underperformer'
    elif percentile >= HIGH_PCT:
        return 'Strong Performer'
    else:
        return 'Normal'

product_signals['signal_v2'] = product_signals.apply(assign_signal_v2, axis=1)

print('=== Signal distribution (v2) ===')
print(product_signals['signal_v2'].value_counts())

In [ ]:
product_df_v2 = product_signals[[
    'view_sessions', 'cart_sessions', 'purchase_sessions',
    'view_to_cart_%', 'cart_to_purchase_%', 'view_to_purchase_%',
    'path_b_flag',
    'cat_level_1', 'cat_level_2',
    'situation', 'primary_metric', 'primary_percentile',
    'signal_v2',
    'benchmark_level',
]].copy()

product_df_v2 = product_df_v2.rename(columns={
    'view_sessions'      : 'view',
    'cart_sessions'      : 'cart',
    'purchase_sessions'  : 'purchase',
    'view_to_cart_%'     : 'view_to_cart_pct',
    'view_to_purchase_%' : 'view_to_purchase_pct',
    'cart_to_purchase_%' : 'cart_to_purchase_pct',
})

product_df_v2['conversion_rate'] = (product_df_v2['view_to_purchase_pct'] / 100).round(4)
product_df_v2['cart_rate']       = (product_df_v2['view_to_cart_pct'] / 100).round(4)

cat_brand = (
    df_cat[['product_id', 'category_code', 'brand']]
    .drop_duplicates('product_id')
    .set_index('product_id')
)
product_df_v2 = product_df_v2.join(cat_brand, how='left')

product_df_v2 = product_df_v2.join(
    price_with_cat[['median_view_price', 'price_percentile']], how='left'
)
product_df_v2 = product_df_v2.rename(columns={'median_view_price': 'price'})

product_df_v2 = product_df_v2.join(
    product_full[[
        'total_revenue', 'avg_purchase_price', 'purchase_count',
        'revenue_rank_in_category', 'revenue_percentile_in_category'
    ]], how='left'
)
product_df_v2['total_revenue'] = product_df_v2['total_revenue'].fillna(0)
product_df_v2 = product_df_v2.rename(columns={'total_revenue': 'revenue_proxy'})

product_df_v2 = product_df_v2.join(
    product_quad[['quadrant', 'l2_median_cr']], how='left'
)

product_df_v2 = product_df_v2.join(
    category_priority[['path_b_rate', 'v2c_pct', 'c2p_pct', 'v2p_pct']],
    on=['cat_level_1', 'cat_level_2'],
    how='left'
)
product_df_v2 = product_df_v2.rename(columns={
    'v2c_pct': 'cat_v2c_benchmark',
    'c2p_pct': 'cat_c2p_benchmark',
    'v2p_pct': 'cat_v2p_benchmark',
})

product_df_v2 = product_df_v2.join(signal_action_df_v2, on='signal_v2', how='left')

product_df_v2['total_interactions'] = (
    product_df_v2['view'] + product_df_v2['cart'] + product_df_v2['purchase']
)

product_df_v2['benchmark_cr'] = np.where(
    product_df_v2['primary_metric'] == 'view_to_cart_%',
    product_df_v2['cat_v2c_benchmark'] / 100,
    np.where(
        product_df_v2['primary_metric'] == 'cart_to_purchase_%',
        product_df_v2['cat_c2p_benchmark'] / 100,
        product_df_v2['cat_v2p_benchmark'] / 100  # Situation 2 & 4
    )
)

product_df_v2['cr_gap'] = (
    product_df_v2['benchmark_cr'] - product_df_v2['conversion_rate']
).clip(lower=0)

product_df_v2['lost_revenue_est'] = (
    product_df_v2['view'] *
    product_df_v2['price'].fillna(0) *
    product_df_v2['cr_gap']
).round(2)

_min = product_df_v2['lost_revenue_est'].min()
_max = product_df_v2['lost_revenue_est'].max()
product_df_v2['popularity_score_raw'] = (
    0.45 * product_df_v2['view'] +
    0.80 * product_df_v2['cart'] +
    1.60 * product_df_v2['purchase']
)
product_df_v2['popularity_score'] = (
    100 * (product_df_v2['popularity_score_raw'] - product_df_v2['popularity_score_raw'].min()) /
    (product_df_v2['popularity_score_raw'].max() - product_df_v2['popularity_score_raw'].min())
).round(1)

POPULARITY_HIGH = product_df_v2['popularity_score'].quantile(0.75)
POPULARITY_LOW  = product_df_v2['popularity_score'].quantile(0.25)

product_df_v2['popularity_label'] = product_df_v2['popularity_score'].apply(
    lambda s: 'High'   if s >= POPULARITY_HIGH
         else 'Low'    if s <= POPULARITY_LOW
         else 'Medium'
)

print(f'product_df_v2 rows: {len(product_df_v2):,}  |  cols: {len(product_df_v2.columns)}')

---
## 8. Assemble & Export Dashboard Tables

This section assembles the three tables consumed by the Streamlit dashboard and exports them to the Databricks Volume.

| File | Level | Primary audience |
|---|---|---|
| `dashboard_products_v2.csv` | One row per product | Seller |
| `dashboard_category_v2.csv` | One row per category | Platform operator |
| `dashboard_brand_v2.csv` | One row per brand × category | Seller |

See `DATA_DICTIONARY.md` for full column definitions for each file.


In [ ]:
ACTIONABLE_SIGNALS = [
    'Low Cart Rate — fix product page',
    'Low Direct Purchase Rate — fix visibility',
    'Low Checkout Rate — fix checkout',
    'Low Overall CR — review traffic quality',
    'High Direct Purchase (Path B)',
]

actionable_products = product_df_v2[
    product_df_v2['signal_v2'].isin(ACTIONABLE_SIGNALS)
].copy()

cat_actionable = (
    actionable_products
    .groupby(['cat_level_1', 'cat_level_2'])
    .agg(
        actionable_product_count     = ('signal_v2', 'count'),
        actionable_lost_revenue      = ('lost_revenue_est', 'sum'),
        avg_lost_revenue_per_product = ('lost_revenue_est', 'mean'),
    )
    .round(2)
)

signal_dist = (
    actionable_products
    .groupby(['cat_level_1', 'cat_level_2', 'signal_v2'])
    .size()
    .unstack(fill_value=0)
)

cat_context = category_priority[[
    'situation', 'primary_metric', 'path_b_rate',
    'v2c_pct', 'c2p_pct',
    'view_sessions',
    'priority_focus', 'action',
]].copy()

total_product_count = (
    product_df_v2
    .groupby(['cat_level_1', 'cat_level_2'])
    .size()
    .rename('total_product_count')
)

category_table = (
    cat_context
    .join(total_product_count, how='left')
    .join(cat_actionable, how='left')
    .join(signal_dist, how='left')
    .fillna(0)
)

category_table['actionable_rate'] = (
    category_table['actionable_product_count'] /
    category_table['total_product_count'] * 100
).round(1)

# ── FIXED sort: rank by v2c benchmark gap (worst relative performers first)
# This surfaces categories that are genuinely underperforming vs their peers,
# not just the ones that happen to sell expensive products.
cat_median_v2c = category_table['v2c_pct'].median()
category_table['v2c_benchmark_gap'] = (cat_median_v2c - category_table['v2c_pct']).round(2)

category_table = category_table.sort_values('v2c_benchmark_gap', ascending=False)
category_table['priority_rank'] = range(1, len(category_table) + 1)

print('=== Category table ===')
print(f'Rows: {len(category_table)}')
print()
print(category_table[[
    'priority_rank', 'situation', 'v2c_pct', 'v2c_benchmark_gap',
    'c2p_pct', 'path_b_rate', 'actionable_product_count', 'actionable_lost_revenue'
]].to_string())
print()
print('=== Situation distribution ===')
print(category_table['situation'].value_counts())

In [ ]:
EXPORT_PATH = '/Volumes/msbabigdata/spark/trend_market'

category_out = f'{EXPORT_PATH}/dashboard_category_v2.csv'
category_table.reset_index().to_csv(category_out, index=False)
print(f'✓ dashboard_category_v2.csv saved  ({len(category_table):,} rows)')

product_out_v2 = f'{EXPORT_PATH}/dashboard_products_v2.csv'
product_df_v2.reset_index().to_csv(product_out_v2, index=False)
print(f'✓ dashboard_products_v2.csv saved  ({len(product_df_v2):,} rows)')

In [ ]:

brand_actionable = actionable_products[
    actionable_products['brand'].notna()
].copy()

brand_agg = (
    brand_actionable
    .groupby(['cat_level_2', 'brand'])
    .agg(
        actionable_product_count      = ('signal_v2', 'count'),
        total_actionable_lost_revenue = ('lost_revenue_est', 'sum'),
        avg_lost_revenue_per_product  = ('lost_revenue_est', 'mean'),
    )
    .round(2)
)

brand_signal_dist = (
    brand_actionable
    .groupby(['cat_level_2', 'brand', 'signal_v2'])['lost_revenue_est']
    .sum()
    .unstack(fill_value=0)
)

brand_signal_dist['dominant_signal'] = brand_signal_dist[
    [c for c in brand_signal_dist.columns if c in ACTIONABLE_SIGNALS]
].idxmax(axis=1)

brand_table = (
    brand_agg
    .join(brand_signal_dist[['dominant_signal']], how='left')
)

brand_table = brand_table.join(
    signal_action_df_v2[['priority_focus_v2']],
    on='dominant_signal',
    how='left'
)

cat_situation = (
    category_priority[['situation', 'primary_metric', 'path_b_rate']]
    .reset_index(level='cat_level_1', drop=True)
)
cat_situation = cat_situation[~cat_situation.index.duplicated(keep='first')]

brand_table = brand_table.join(
    cat_situation,
    on='cat_level_2',
    how='left'
)

brand_table_filtered = brand_table[
    brand_table['actionable_product_count'] >= MIN_BRAND_PRODUCTS
].copy()

brand_table_filtered['action_type'] = brand_table_filtered['dominant_signal'].map({
    'High Direct Purchase (Path B)'             : 'Scale',
    'Low Cart Rate — fix product page'          : 'Fix',
    'Low Direct Purchase Rate — fix visibility' : 'Fix',
    'Low Checkout Rate — fix checkout'          : 'Fix',
    'Low Overall CR — review traffic quality'   : 'Fix',
})

brand_table_filtered['rank_in_category'] = (
    brand_table_filtered
    .groupby('cat_level_2')['total_actionable_lost_revenue']
    .rank(ascending=False, method='dense')
    .astype(int)
)

print(f'Brand × category combinations: {len(brand_table_filtered):,}')
print()
print('=== Action type distribution ===')
print(brand_table_filtered['action_type'].value_counts())
print()
print('=== Situation distribution ===')
print(brand_table_filtered['situation'].value_counts())

In [ ]:
brand_table_filtered = brand_table_filtered.reset_index()

# 1. 問題一句話
situation_summary = {
    'Situation 1: Cart barrier':               'View→Cart rate is below category average — product page may need improvement.',
    'Situation 2: Discovery/appeal problem':   'Overall purchase rate is low — products may lack visibility or appeal.',
    'Situation 3: Checkout friction':          'Cart→Purchase rate is below category average — pricing or shipping may be blocking buyers.',
    'Situation 4: Traffic quality problem':    'Overall conversion is very low — traffic may not be reaching the right audience.',
    'Healthy':                                 'Both funnel metrics are above platform average.',
}
brand_table_filtered['situation_summary'] = brand_table_filtered['situation'].map(situation_summary)

# 2. Brand 在這個 category 裡的相對排名
brand_table_filtered['brand_rank_in_category'] = (
    brand_table_filtered
    .groupby('cat_level_2')['total_actionable_lost_revenue']
    .rank(ascending=False, method='dense')
    .astype(int)
)

brand_table_filtered['brands_in_category'] = (
    brand_table_filtered
    .groupby('cat_level_2')['brand']
    .transform('count')
)

brand_table_filtered['brand_rank_label'] = (
    brand_table_filtered['brand_rank_in_category'].astype(str) +
    ' of ' +
    brand_table_filtered['brands_in_category'].astype(str) +
    ' brands'
)

print(brand_table_filtered[['brand', 'cat_level_2', 'situation', 'situation_summary', 'brand_rank_label']].head(10).to_string())

In [ ]:
EXPORT_PATH = '/Volumes/msbabigdata/spark/trend_market'

brand_out = f'{EXPORT_PATH}/dashboard_brand_v2.csv'
brand_table_filtered.to_csv(brand_out, index=False)
print(f'✓ dashboard_brand_v2.csv saved  ({len(brand_table_filtered):,} rows)')